# Tire Analysis: Temperature Distribution & Thermography

This notebook visualizes tire temperature data from infrared sensors across your tires, helping you understand tire behavior and identify setup issues.

## What You'll Find Here

- **Tire Temperature Heatmaps**: Visual representation of temperature distribution across all four tires (FL, FR, RL, RR) throughout the lap
- **Temperature vs Distance**: See how tire temperatures change through different track sections
- **Speed & G-Force Overlay**: Correlate tire temperatures with speed and combined lateral/longitudinal acceleration
- **Driver Inputs Overlay**: See throttle, brake, and steering inputs aligned with temperature data

## How to Interpret the Results

### Temperature Heatmaps
- **Ch1-Ch8**: Represent temperature sensor positions across the tire width
  - FL/RL: Ch1 = Outside (left), Ch8 = Inside (right)
  - FR/RR: Ch1 = Inside (left), Ch8 = Outside (right)
- **Hot spots (bright)**: Areas of high temperature - could indicate excessive load or slip
- **Cold spots (dark)**: Areas not being worked - potential grip left on the table
- **Even temperature gradient**: Indicates good tire usage and camber settings

### Setup Insights
- **Outside edge hot**: May need more negative camber
- **Inside edge hot**: May have too much negative camber
- **Center hot**: Could indicate over-inflation
- **Edges hot, center cold**: Could indicate under-inflation
- **Front vs Rear difference**: Balance insights for understeer/oversteer tendencies

## Using Your Own Data

To analyze your own data:

1. **Run the file picker cell** below to display the upload widget
2. **Drag and drop** your `.xrk` or `.xrz` file onto the upload button (or click to browse)
3. **Run all remaining cells** to analyze your data

The status indicator will show which file is being used. If you don't upload a file, the sample data will be used.

## Requirements

- Tire temperature channels (`FL_Ch1`-`FL_Ch8`, `FR_Ch1`-`FR_Ch8`, `RL_Ch1`-`RL_Ch8`, `RR_Ch1`-`RR_Ch8`)
- GPS Speed channel for distance calculation
- Acceleration data (`LateralAcc`, `InlineAcc`) for G-force visualization
- Driver input channels (`BrakePress`, `PPS`, `SteerAngle`)

**Note:** This notebook works in both JupyterLite (browser) and standard JupyterLab environments.

In [1]:
# Install required packages (needed for JupyterLite, skipped in regular JupyterLab if already installed)
%pip install -q pandas plotly libxrk motorsports-data-notebook jinja2 ipywidgets

In [2]:
# Import core libraries
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Import libxrk:
from libxrk import aim_xrk

# Import helper functions
from motorsports_data_notebook import show_fig, get_best_lap, compute_lap_distance, FileUpload

In [3]:
# File picker - upload your own .xrk/.xrz file or use the sample data
file_upload = FileUpload(default_file="CMD_Inferno 86_Fuji GP Sh_Generic testing_a_2248.xrz")
file_upload.display()

In [ ]:
# Load the data file
log = aim_xrk(file_upload.get_file_data())

In [4]:
# Flatten all the data into a uniform table, interpolating as needed (can use a lot of RAM!)
channels = log.get_channels_as_table().to_pandas()

# Add derived columns
channels["speed_kmh"] = channels["GPS Speed"] * 3.6

In [5]:
# Load laps and compute lap times
laps = log.laps.to_pandas()
laps["lap_time"] = pd.to_timedelta(laps["end_time"] - laps["start_time"], unit="ms")

# Compute distance_m for each lap and add to channels
# Distance resets at the start of each lap
channels["distance_m"] = 0.0

for idx, lap in laps.iterrows():
    lap_mask = (channels["timecodes"] >= lap["start_time"]) & (
        channels["timecodes"] <= lap["end_time"]
    )
    lap_indices = channels.index[lap_mask]

    if len(lap_indices) > 0:
        lap_timecodes = channels.loc[lap_indices, "timecodes"]
        lap_speed = channels.loc[lap_indices, "GPS Speed"]
        distance_values = compute_lap_distance(lap_timecodes.values, lap_speed.values)
        channels.loc[lap_indices, "distance_m"] = distance_values

laps.style.format(
    {"lap_time": lambda x: f"{int(x.total_seconds() // 60)}:{x.total_seconds() % 60:06.3f}"}
)

In [6]:
# Best lap extraction
best_lap = get_best_lap(laps)
start_ts = best_lap["start_time"]
end_ts = best_lap["end_time"]
# Use < for end_ts to exclude the first sample of the next lap (where distance resets to 0)
lap_channels = channels.query(f"timecodes >= @start_ts and timecodes < @end_ts").copy()

In [7]:
# Tire Thermography - Best Lap
# Extract FL, FR, RL, RR tire temperature channels
# Ch1 = leftmost (outside for FL/RL, inside for FR/RR), Ch8 = rightmost (inside for FL/RL, outside for FR/RR)
fl_channels = ["FL_Ch1", "FL_Ch2", "FL_Ch3", "FL_Ch4", "FL_Ch5", "FL_Ch6", "FL_Ch7", "FL_Ch8"]
fr_channels = ["FR_Ch1", "FR_Ch2", "FR_Ch3", "FR_Ch4", "FR_Ch5", "FR_Ch6", "FR_Ch7", "FR_Ch8"]
rl_channels = ["RL_Ch1", "RL_Ch2", "RL_Ch3", "RL_Ch4", "RL_Ch5", "RL_Ch6", "RL_Ch7", "RL_Ch8"]
rr_channels = ["RR_Ch1", "RR_Ch2", "RR_Ch3", "RR_Ch4", "RR_Ch5", "RR_Ch6", "RR_Ch7", "RR_Ch8"]

fl_temps = lap_channels[fl_channels].values.T  # Shape: (8 channels, n_samples)
fr_temps = lap_channels[fr_channels].values.T
rl_temps = lap_channels[rl_channels].values.T
rr_temps = lap_channels[rr_channels].values.T

# Use pre-computed distance_m from lap_channels
distance_m = lap_channels["distance_m"]

# Calculate Sum of G (Euclidean sum of lateral and inline accelerations)
sum_of_g = np.sqrt(lap_channels["LateralAcc"] ** 2 + lap_channels["InlineAcc"] ** 2)

# Get color scale range across all tires for consistent coloring
vmin = min(fl_temps.min(), fr_temps.min(), rl_temps.min(), rr_temps.min())
vmax = max(fl_temps.max(), fr_temps.max(), rl_temps.max(), rr_temps.max())

# Create subplots with shared x-axis (6 rows: 4 tire heatmaps + speed/G plot + inputs plot)
fig = make_subplots(
    rows=6,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.04,
    subplot_titles=(
        "Front Left (Outside at top)",
        "Front Right (Outside at bottom)",
        "Rear Left (Outside at top)",
        "Rear Right (Outside at bottom)",
        "Speed & Sum of G",
        "Driver Inputs",
    ),
    row_heights=[0.17, 0.17, 0.17, 0.17, 0.16, 0.16],
    specs=[[{}], [{}], [{}], [{}], [{"secondary_y": True}], [{"secondary_y": True}]],
)

# Y-axis labels (just channel numbers, no prefix)
y_labels = ["1", "2", "3", "4", "5", "6", "7", "8"]

# Front Left heatmap - FL Ch1 (outside/left) at top
fig.add_trace(
    go.Heatmap(
        z=fl_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale="Inferno",
        zmin=vmin,
        zmax=vmax,
        showscale=False,
    ),
    row=1,
    col=1,
)
fig.update_yaxes(autorange="reversed", row=1, col=1)

# Front Right heatmap - FR Ch8 (outside/right) at bottom
fig.add_trace(
    go.Heatmap(
        z=fr_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale="Inferno",
        zmin=vmin,
        zmax=vmax,
        showscale=False,
    ),
    row=2,
    col=1,
)
fig.update_yaxes(autorange="reversed", row=2, col=1)

# Rear Left heatmap - RL Ch1 (outside/left) at top
fig.add_trace(
    go.Heatmap(
        z=rl_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale="Inferno",
        zmin=vmin,
        zmax=vmax,
        showscale=False,
    ),
    row=3,
    col=1,
)
fig.update_yaxes(autorange="reversed", row=3, col=1)

# Rear Right heatmap - RR Ch8 (outside/right) at bottom
fig.add_trace(
    go.Heatmap(
        z=rr_temps,
        x=distance_m.values,
        y=y_labels,
        colorscale="Inferno",
        zmin=vmin,
        zmax=vmax,
        colorbar=dict(title="Temp (°C)"),
    ),
    row=4,
    col=1,
)
fig.update_yaxes(autorange="reversed", row=4, col=1)

# Speed line plot at row 5 (primary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels["speed_kmh"].values,
        mode="lines",
        name="Speed",
        line=dict(color="black", width=1),
    ),
    row=5,
    col=1,
    secondary_y=False,
)

# Sum of G line plot (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=sum_of_g.values,
        mode="lines",
        name="Sum of G",
        line=dict(color="red", width=1),
    ),
    row=5,
    col=1,
    secondary_y=True,
)

# Driver Inputs subplot (row 6)
# Brake Pressure - red (primary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels["BrakePress"].values,
        mode="lines",
        name="Brake",
        line=dict(color="red", width=1),
    ),
    row=6,
    col=1,
    secondary_y=False,
)

# Throttle (PPS) - green (primary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels["PPS"].values,
        mode="lines",
        name="Throttle",
        line=dict(color="green", width=1),
    ),
    row=6,
    col=1,
    secondary_y=False,
)

# Steering Angle - black (secondary y-axis)
fig.add_trace(
    go.Scatter(
        x=distance_m.values,
        y=lap_channels["SteerAngle"].values,
        mode="lines",
        name="Steering",
        line=dict(color="black", width=1),
    ),
    row=6,
    col=1,
    secondary_y=True,
)

fig.update_layout(
    title="Tire Temperatures - Best Lap",
    xaxis6_title="Distance (m)",
    yaxis_title="FL",
    yaxis2_title="FR",
    yaxis3_title="RL",
    yaxis4_title="RR",
    width=900,
    height=900,
    showlegend=False,
)

# Set y-axis titles for the speed/G subplot (row 5)
fig.update_yaxes(title_text="km/h", row=5, col=1, secondary_y=False)
fig.update_yaxes(title_text="G", row=5, col=1, secondary_y=True)

# Set y-axis titles for the driver inputs subplot (row 6)
fig.update_yaxes(title_text="%", row=6, col=1, secondary_y=False)
fig.update_yaxes(title_text="deg", row=6, col=1, secondary_y=True)

# Hide tick labels on y-axes for heatmaps (keep only the axis title)
fig.update_yaxes(showticklabels=False, row=1, col=1)
fig.update_yaxes(showticklabels=False, row=2, col=1)
fig.update_yaxes(showticklabels=False, row=3, col=1)
fig.update_yaxes(showticklabels=False, row=4, col=1)

show_fig(fig)